# Notebook 1: Fetch NBA Data
Pulls 2025 NBA player box score data using `nba_api` and saves it as CSV files.

In [1]:

import pandas as pd
import time
import os
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv3
from nba_api.stats.static import players, teams
import random

os.makedirs('data', exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


## Step 1: Fetch All Games for 2024-25 Season

In [2]:
# Fetch regular season game log
gamelog_rs = leaguegamelog.LeagueGameLog(
    season='2024-25',
    season_type_all_star='Regular Season'
).get_data_frames()[0]

# Fetch playoffs game log
gamelog_po = leaguegamelog.LeagueGameLog(
    season='2024-25',
    season_type_all_star='Playoffs'
).get_data_frames()[0]

# Combine and tag season type
gamelog_rs['SEASON_TYPE'] = 'Regular Season'
gamelog_po['SEASON_TYPE'] = 'Playoffs'
all_games = pd.concat([gamelog_rs, gamelog_po], ignore_index=True)

print(f'Total team-game records: {len(all_games)}')
all_games.head()

Total team-game records: 2628


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON_TYPE
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,40,33,6,3,4,15,132,23,1,Regular Season
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,47,17,4,1,16,22,103,-7,1,Regular Season
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,46,22,7,8,7,22,110,7,1,Regular Season
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,34,20,2,3,12,12,109,-23,1,Regular Season
4,22024,1610612744,GSW,Golden State Warriors,0022400072,2024-10-23,GSW @ POR,W,240,48,...,57,38,13,5,18,27,140,36,1,Regular Season


## Step 2: Build Games Table

In [3]:
# Deduplicate to one row per game
games_df = all_games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'SEASON_TYPE']].drop_duplicates(subset='GAME_ID').copy()

# Parse home/away from MATCHUP (e.g. 'GSW vs. LAL' or 'GSW @ LAL')
def parse_matchup(row):
    if 'vs.' in row['MATCHUP']:
        home = row['MATCHUP'].split(' vs. ')[0].strip()
        away = row['MATCHUP'].split(' vs. ')[1].strip()
    else:
        away = row['MATCHUP'].split(' @ ')[0].strip()
        home = row['MATCHUP'].split(' @ ')[1].strip()
    return home, away

games_df[['HOME_TEAM', 'AWAY_TEAM']] = games_df.apply(
    lambda r: pd.Series(parse_matchup(r)), axis=1
)

games_df = games_df[['GAME_ID', 'GAME_DATE', 'HOME_TEAM', 'AWAY_TEAM', 'SEASON_TYPE']]
games_df.to_csv('data/games.csv', index=False)
print(f'Games saved: {len(games_df)}')
games_df.head()

Games saved: 1314


,GAME_ID,GAME_DATE,HOME_TEAM,AWAY_TEAM,SEASON_TYPE
0,0022400061,2024-10-22,BOS,NYK,Regular Season
1,0022400062,2024-10-22,LAL,MIN,Regular Season
4,0022400072,2024-10-23,POR,GSW,Regular Season
5,0022400068,2024-10-23,HOU,CHA,Regular Season
6,0022400071,2024-10-23,LAC,PHX,Regular Season


## Step 3: Fetch Player Box Scores
> ⚠️ This step makes one API call per game and includes a short delay to avoid rate limiting. It may take 20–40 minutes for the full season.

In [ ]:
game_ids = games_df['GAME_ID'].unique().tolist()
all_boxscores = []
failed = []

for i, gid in enumerate(game_ids):
    try:
        bs = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=gid, timeout = 60)
        df = bs.get_data_frames()[0]
        all_boxscores.append(df)
        if (i + 1) % 50 == 0:
            print(f'Fetched {i+1}/{len(game_ids)} games...')
        time.sleep(random.randint(1,5)) 
    except Exception as e:
        print(f'Failed game {gid}: {e}')
        failed.append(gid)
        time.sleep(random.randint(1,5))

boxscores_df = pd.concat(all_boxscores, ignore_index=True)
print(f'Total player-game records: {len(boxscores_df)}')
if failed:
    print(f'Failed game IDs: {failed}')

Fetched 50/1314 games...
Fetched 100/1314 games...
Fetched 150/1314 games...
Fetched 200/1314 games...
Fetched 250/1314 games...
Fetched 300/1314 games...
Fetched 350/1314 games...


## Step 4: Build PlayerBoxScores and Players Tables

In [ ]:
# Select and rename relevant columns
box_cols = [
    'GAME_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION',
    'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT', 'PLUS_MINUS'
]
player_boxscores = boxscores_df[box_cols].copy()

# Merge in season type and game date from games table
player_boxscores = player_boxscores.merge(
    games_df[['GAME_ID', 'GAME_DATE', 'SEASON_TYPE']],
    on='GAME_ID', how='left'
)

player_boxscores.to_csv('data/player_boxscores.csv', index=False)
print(f'PlayerBoxScores saved: {len(player_boxscores)} rows')

# Build Players table
players_df = player_boxscores[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION']].drop_duplicates(subset='PLAYER_ID')
players_df.columns = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM']
players_df.to_csv('data/players.csv', index=False)
print(f'Players saved: {len(players_df)} unique players')
player_boxscores.head()